# 01 – Datensatz aufbauen: Roh-CSV → `data_final.parquet`

Dieses Notebook dokumentiert die vollständige Pipeline zur Erstellung von `data/processed/data_final.parquet`.

**Wenn `data_final.parquet` bereits existiert:** Die Zellen 2–12 können übersprungen werden.  
Direkt zu **Schritt 13 (Validierung)** springen um den bestehenden Datensatz zu prüfen.

**Pipeline (zur Reproduktion):**
1. Roh-CSV laden + Koordinaten dekodieren
2. Wetterdaten fetchen (Open-Meteo API, checkpoint-basiert)
3. Wetter auf 15-min-PV-Daten mergen
4. Solar-Geometrie berechnen (pvlib)
5. Einstrahlungs-Regime-Features
6. Rolling-Features (Schnee, Niederschlag, Temperatur)
7. Zeit-Features (zyklisch)
8. Panel-Tilt/Azimuth aus SPT + Inference für fehlende Spots
9. kwp\_est + kwh\_norm
10. Analog-Features (k=10, saisonale Nähe +-60 Tage)
11. Outlier-Entfernung
12. Feature-Auswahl & Speichern
13. Validierung


## 0. Konfiguration & Imports

In [ ]:
import os, sys, gc, time, struct, binascii, warnings
import numpy as np
import pandas as pd
import requests
import pvlib
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

warnings.filterwarnings('ignore')

# ── Pfade ─────────────────────────────────────────────────────────────────────
_HERE = Path.cwd()
_BASE = _HERE / 'bscthesis' if (_HERE / 'bscthesis').exists() else _HERE

RAW_CSV        = _BASE / 'data/raw/grid_feed_in_15min_202603271142.csv'
META_CSV       = _BASE / 'data/raw/location_meta_202603250933.csv'
RECENT_PARQUET = (_HERE / 'experiments/total_pv_output/data/processed/2026-03-03_pv_with_analogs.parquet'
                  if (_HERE / 'experiments').exists() else
                  _BASE.parent / 'experiments/total_pv_output/data/processed/2026-03-03_pv_with_analogs.parquet')
OUT_PARQUET    = _BASE / 'data/processed/data_final.parquet'
CP_DIR         = _BASE / 'data/processed/weather_checkpoints_15min'  # 15-min Checkpoints
INTERMEDIATE   = _BASE / 'data/processed/hist_pv_intermediate.parquet'
CP_DIR.mkdir(parents=True, exist_ok=True)

# ── Einstellungen ─────────────────────────────────────────────────────────────
COMPUTE_ANALOGS = True
N_WORKERS_TILT  = -2
OPEN_METEO_API_KEY = os.getenv('OPEN_METEO_API_KEY', '')

SEASON_PROXIMITY_DAYS     = 60
SOLAR_ELEVATION_THRESHOLD = 3.0
EPS = 1e-6

# minutely_15: native 15-min Modelldaten für Zentraleuropa (DWD ICON-D2)
# Andere Variablen werden von Open-Meteo intern von stündlich auf 15-min interpoliert
WEATHER_VARS = [
    'shortwave_radiation', 'diffuse_radiation', 'direct_normal_irradiance',
    'temperature_2m', 'precipitation', 'wind_speed_10m',
    'snowfall', 'visibility', 'relative_humidity_2m', 'weather_code',
]
RENAME_MAP = {
    'shortwave_radiation':      'ghi',
    'diffuse_radiation':        'dhi',
    'direct_normal_irradiance': 'dni',
    'precipitation':            'precipitation_mm',
}
WEATHER_REGIME_MAP = {
    0: 'clear', 1: 'partly_cloudy', 2: 'partly_cloudy', 3: 'overcast',
    45: 'fog', 48: 'fog',
    51: 'drizzle', 53: 'drizzle', 55: 'drizzle',
    56: 'freezing_precip', 57: 'freezing_precip',
    61: 'rain', 63: 'rain', 65: 'rain',
    66: 'freezing_precip', 67: 'freezing_precip',
    71: 'snow', 73: 'snow', 75: 'snow', 77: 'snow',
    80: 'rain', 81: 'rain', 82: 'rain',
    85: 'snow', 86: 'snow',
    95: 'thunderstorm', 96: 'thunderstorm', 99: 'thunderstorm',
}
ORDERED_CATEGORIES = ['clear','partly_cloudy','overcast','fog',
                      'drizzle','rain','freezing_precip','snow','thunderstorm']

FINAL_FEATURES = [
    'ts', 'spot_uuid', 'kwh', 'kwh_norm',
    'ghi', 'dhi', 'dni', 'tsi', 'kt',
    'dni_frac', 'dhi_frac', 'ghi_magnitude',
    'solar_elevation', 'solar_azimuth',
    'temperature_2m', 'precipitation_mm', 'wind_speed_10m',
    'relative_humidity_2m', 'snowfall', 'visibility',
    'temperature_penalty_rel', 'weather_regime',
    'snow_roll6h', 'snow_roll24h', 'precip_roll6h', 'temp_roll24h',
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'doy_sin', 'doy_cos',
    'latitude', 'longitude', 'panel_tilt',
    'max_kwh_est', 'kwp_est', 'panel_unimodal_azimuth', 'panel_tilt_azi_confidence',
    'panel_bimodal_east_azimuth', 'panel_bimodal_east_fraction',
    'panel_bimodal_west_azimuth', 'panel_is_ew_split', 'irr_mismatch',
    'analog_kwh_1', 'analog_kwh_1_norm', 'analog_knn_mean', 'analog_knn_mean_norm',
    'scaled_analog_kwh_1', 'scaled_analog_kwh_1_norm', 'scaled_analog_knn_mean',
    'analog_available', 'analog_dist_1', 'analog_dist_gap', 'analog_knn_std', 'analog_age_hours',
]

print(f'RAW_CSV        : {RAW_CSV}  exists={RAW_CSV.exists()}')
print(f'META_CSV       : {META_CSV}  exists={META_CSV.exists()}')
print(f'RECENT_PARQUET : {RECENT_PARQUET}  exists={RECENT_PARQUET.exists()}')
print(f'OUT_PARQUET    : {OUT_PARQUET}')
print(f'INTERMEDIATE   : {INTERMEDIATE}  exists={INTERMEDIATE.exists()}')


In [ ]:
# ── Existiert data_final.parquet schon? ──────────────────────────────────────
if OUT_PARQUET.exists():
    size_gb = OUT_PARQUET.stat().st_size / 1e9
    print(f'data_final.parquet existiert bereits ({size_gb:.2f} GB).')
    print('Schritte 1-12 können übersprungen werden → direkt zu Zelle 13 (Validierung).')
    print()
    print(f'Vorhandene Zwischendateien:')
    print(f'  INTERMEDIATE : {INTERMEDIATE}  exists={INTERMEDIATE.exists()}')
    print(f'  CP_DIR       : {CP_DIR}  exists={CP_DIR.exists()}, '
          f'{len(list(CP_DIR.glob("*.parquet")))} Checkpoints')
    print(f'  RECENT_PARQUET: {RECENT_PARQUET}  exists={RECENT_PARQUET.exists()}')
else:
    print('data_final.parquet nicht gefunden – Schritte 1-12 ausführen.')

## 1. Roh-CSV laden + Koordinaten dekodieren

In [ ]:
def decode_ewkb(h):
    b = binascii.unhexlify(h)
    return struct.unpack_from('<d', b, 17)[0], struct.unpack_from('<d', b, 9)[0]

meta = pd.read_csv(META_CSV)
meta[['latitude', 'longitude']] = meta['address_coordinates'].apply(
    lambda x: pd.Series(decode_ewkb(x)))
meta = meta[['spot_uuid', 'latitude', 'longitude']].drop_duplicates('spot_uuid')

# Koordinaten aus recent Parquet ergaenzen (praeziser fuer bekannte Spots)
if RECENT_PARQUET.exists():
    try:
        recent_meta = pd.read_parquet(RECENT_PARQUET,
            columns=['spot_uuid', 'latitude', 'longitude']).drop_duplicates('spot_uuid')
        meta = meta.merge(recent_meta.rename(columns={'latitude':'_lat2','longitude':'_lon2'}),
                          on='spot_uuid', how='left')
        meta['latitude']  = meta['_lat2'].combine_first(meta['latitude'])
        meta['longitude'] = meta['_lon2'].combine_first(meta['longitude'])
        meta = meta[['spot_uuid', 'latitude', 'longitude']]
        print(f'Koordinaten aus recent Parquet ergaenzt.')
    except Exception as e:
        print(f'recent Koordinaten nicht geladen: {e}')

df_raw = pd.read_csv(RAW_CSV, parse_dates=False)
df_raw.columns = df_raw.columns.str.strip()
df_raw = df_raw.rename(columns={'bucket_15min': 'ts'})
df_raw['ts']  = pd.to_datetime(df_raw['ts'], utc=True).dt.tz_localize(None)
df_raw['kwh'] = pd.to_numeric(df_raw['kwh'], errors='coerce').fillna(0.0)
df_raw = df_raw.merge(meta, on='spot_uuid', how='inner').dropna(subset=['latitude','longitude'])
df_raw = df_raw.sort_values(['spot_uuid','ts']).reset_index(drop=True)

start_str = df_raw['ts'].min().strftime('%Y-%m-%d')
end_str   = df_raw['ts'].max().strftime('%Y-%m-%d')
print(f'Geladen: {len(df_raw):,} Zeilen | {df_raw["spot_uuid"].nunique()} Spots')
print(f'Zeitraum: {start_str} bis {end_str}')


## 2. Wetterdaten fetchen (Open-Meteo, 15-min, checkpoint-basiert)

Fuer jeden Spot wird ein **15-minütiger** Wetter-Checkpoint gespeichert (`minutely_15` Parameter).
Für Zentraleuropa (Deutschland) liefert Open-Meteo native 15-min-Modelldaten (DWD ICON-D2).
Bereits gefetchte Spots werden uebersprungen — Unterbrechung jederzeit moeglich.

Optional: `OPEN_METEO_API_KEY` als Umgebungsvariable setzen fuer hoehere Rate-Limits.

In [ ]:
def fetch_weather(lat, lon, start, end):
    base = ('https://customer-historical-forecast-api.open-meteo.com/v1/forecast'
            if OPEN_METEO_API_KEY else
            'https://historical-forecast-api.open-meteo.com/v1/forecast')
    params = {'latitude': lat, 'longitude': lon, 'minutely_15': ','.join(WEATHER_VARS),
              'start_date': start, 'end_date': end, 'timezone': 'UTC'}
    if OPEN_METEO_API_KEY:
        params['apikey'] = OPEN_METEO_API_KEY
    for attempt in range(3):
        try:
            r = requests.get(base, params=params, timeout=30); r.raise_for_status()
            wdf = pd.DataFrame(r.json()['minutely_15']).rename(columns={'time': 'ts'})
            wdf['ts'] = pd.to_datetime(wdf['ts'])
            return wdf.rename(columns=RENAME_MAP)
        except Exception as e:
            if attempt == 2: raise
            time.sleep(3 * (attempt + 1))

spot_locs  = df_raw.groupby('spot_uuid')[['latitude','longitude']].first().reset_index()
done       = {f.stem for f in CP_DIR.glob('*.parquet')}
remaining  = spot_locs[~spot_locs['spot_uuid'].isin(done)]
print(f'Spots gesamt: {len(spot_locs)} | bereits gecacht: {len(done)} | offen: {len(remaining)}')

errors, errors_lock = [], threading.Lock()

def fetch_one(row):
    cp = CP_DIR / f'{row["spot_uuid"]}.parquet'
    if cp.exists(): return None
    try:
        wdf = fetch_weather(row['latitude'], row['longitude'], start_str, end_str)
        wdf['spot_uuid'] = row['spot_uuid']
        wdf.to_parquet(cp, index=False)
        return None
    except Exception as e:
        with errors_lock: errors.append(row['spot_uuid'])
        return f'FEHLER {row["spot_uuid"][:8]}: {e}'

if len(remaining) > 0:
    with ThreadPoolExecutor(max_workers=6) as ex:
        futs = {ex.submit(fetch_one, r): r for _, r in remaining.iterrows()}
        done_n = 0
        for fut in as_completed(futs):
            err = fut.result(); done_n += 1
            if err: print(f'  {err}')
            elif done_n % 50 == 0: print(f'  {done_n}/{len(remaining)} gefetcht...')

print(f'Fetch abgeschlossen. Fehler: {len(errors)}')
if errors: print(f'  Fehlgeschlagen: {errors[:5]}')


## 3–8. Intermediate-Checkpoint aufbauen

Diese Schritte werden **übersprungen** wenn `hist_pv_intermediate.parquet` bereits existiert.

In [ ]:
# Intermediate-Flag: True = Schritte 3-8 müssen ausgeführt werden
_BUILD = not INTERMEDIATE.exists()
if _BUILD:
    print('Baue Intermediate Checkpoint (Schritte 3-8)...')
else:
    print(f'Checkpoint gefunden: {INTERMEDIATE} -- Schritte 3-8 werden übersprungen.')

## 3. Wetter auf 15-min-PV-Daten mergen

Die 15-minütigen Wetter-Checkpoints werden per exaktem Timestamp-Join (`merge` auf `['ts', 'spot_uuid']`) auf die PV-Zeitreihe gemergt — kein Nearest-Neighbor mehr nötig, da beide Datensätze auf 15-min-Basis vorliegen.

In [ ]:
if _BUILD:
    print('  Lade 15-min Wetter-Checkpoints...')
    weather_15min = pd.concat(
        [pd.read_parquet(f) for f in CP_DIR.glob('*.parquet')], ignore_index=True)
    weather_15min['ts'] = pd.to_datetime(weather_15min['ts'])
    df = df_raw.merge(weather_15min, on=['ts', 'spot_uuid'], how='left')
    del df_raw, weather_15min; gc.collect()
    float_cols = df.select_dtypes('float64').columns
    df[float_cols] = df[float_cols].astype('float32')
    print(f'  Merge: {len(df):,} Zeilen | {df["spot_uuid"].nunique()} Spots')
    n_missing = df['ghi'].isna().sum()
    print(f'  Fehlende Wetterwerte nach Merge: {n_missing:,} ({n_missing/len(df)*100:.2f}%)')


## 4. Solar-Geometrie (pvlib)

Für jeden Spot und Zeitpunkt: Sonnenhöhe (`solar_elevation`), Sonnenazimut (`solar_azimuth`), Total Solar Irradiance (`tsi`) und Clearness Index `kt = GHI / TOA`. Nachtstunden: `kt = 0`.

In [ ]:
if _BUILD:
    print('  Solar-Geometrie (pvlib)...')
    solar_parts = []
    for uuid, grp in df.groupby('spot_uuid', sort=False):
        sol = pvlib.solarposition.get_solarposition(
            pd.DatetimeIndex(grp['ts']).tz_localize('UTC'),
            grp['latitude'].iloc[0], grp['longitude'].iloc[0])
        solar_parts.append(pd.DataFrame({
            'solar_elevation': sol['apparent_elevation'].values.clip(-90, 90),
            'solar_azimuth':   sol['azimuth'].values}, index=grp.index))
    solar_df = pd.concat(solar_parts)
    df['solar_elevation'] = solar_df['solar_elevation'].astype('float32')
    df['solar_azimuth']   = solar_df['solar_azimuth'].astype('float32')
    df['tsi'] = (df['ghi'].clip(0) + df['dni'].clip(0) + df['dhi'].clip(0)).clip(0)
    toa = (1361.0 * np.cos(np.radians(90 - df['solar_elevation'].clip(0)))).clip(1e-3)
    df['kt'] = (df['ghi'].clip(0) / toa).clip(0, 1.2).astype('float32')
    df.loc[df['solar_elevation'] <= 0, 'kt'] = 0.0
    print(f'  solar_elevation: {df["solar_elevation"].min():.1f}° – {df["solar_elevation"].max():.1f}°')
    print(f'  kt:              {df["kt"].min():.3f} – {df["kt"].max():.3f}')

## 5. Einstrahlungs-Regime-Features

Anteil direkter (`dni_frac`) und diffuser (`dhi_frac`) Strahlung an der Gesamteinstrahlung, sowie normierte GHI-Magnitude.

In [ ]:
if _BUILD:
    total = df['ghi'].clip(0) + df['dni'].clip(0) + df['dhi'].clip(0) + EPS
    df['dni_frac']      = (df['dni'].clip(0) / total).astype('float32')
    df['dhi_frac']      = (df['dhi'].clip(0) / total).astype('float32')
    df['ghi_magnitude'] = (df['ghi'].clip(0) / 1000.0).astype('float32')
    print(f'  dni_frac:      {df["dni_frac"].mean():.3f} (Mittel)')
    print(f'  dhi_frac:      {df["dhi_frac"].mean():.3f} (Mittel)')
    print(f'  ghi_magnitude: {df["ghi_magnitude"].mean():.3f} (Mittel)')

## 6. Rolling-Features (pro Spot)

Kumulierter Schneefall (6h, 24h), Niederschlag (6h) und gleitender Temperaturmittelwert (24h). Berechnung strikt pro Spot – keine Werte überschreiten Spot-Grenzen.

In [ ]:
if _BUILD:
    print('  Rolling-Features...')
    df = df.sort_values(['spot_uuid','ts']).reset_index(drop=True)
    for col, win, name in [('snowfall', 6,  'snow_roll6h'),
                            ('snowfall', 24, 'snow_roll24h'),
                            ('precipitation_mm', 6, 'precip_roll6h')]:
        df[name] = (df.groupby('spot_uuid')[col]
                      .transform(lambda x: x.rolling(win, min_periods=1).sum())
                      .astype('float32'))
    df['temp_roll24h'] = (df.groupby('spot_uuid')['temperature_2m']
                            .transform(lambda x: x.rolling(24, min_periods=1).mean())
                            .astype('float32'))
    print(f'  snow_roll24h:  max={df["snow_roll24h"].max():.2f} cm')
    print(f'  precip_roll6h: max={df["precip_roll6h"].max():.2f} mm')

## 7. Zeit-Features (zyklisch)

Stunde, Monat und Jahrestag als Sinus/Kosinus-Paare, damit das Modell die Zyklizität der Zeit versteht (z.B. Stunde 23 liegt nah bei Stunde 0).

In [ ]:
if _BUILD:
    h = df['ts'].dt.hour + df['ts'].dt.minute / 60
    df['hour_sin']  = np.sin(2*np.pi*h/24).astype('float32')
    df['hour_cos']  = np.cos(2*np.pi*h/24).astype('float32')
    df['month_sin'] = np.sin(2*np.pi*df['ts'].dt.month/12).astype('float32')
    df['month_cos'] = np.cos(2*np.pi*df['ts'].dt.month/12).astype('float32')
    df['doy_sin']   = np.sin(2*np.pi*df['ts'].dt.dayofyear/365).astype('float32')
    df['doy_cos']   = np.cos(2*np.pi*df['ts'].dt.dayofyear/365).astype('float32')
    print(f'  Zyklische Zeit-Features berechnet (6 Spalten).')

## 8. Wetterregime & Temperaturkorrektur

`weather_regime` ordnet den WMO-Wettercode in 9 Kategorien ein (clear, partly_cloudy, overcast, fog, drizzle, rain, freezing_precip, snow, thunderstorm).

`temperature_penalty_rel` bildet die Temperatur auf eine lineare Skala ab, die den Wirkungsgradabfall von Solarmodulen bei hohen Temperaturen widerspiegelt (typischerweise -0.4 %/°C über STC-Referenz von 25°C).

Danach: Zwischenspeicherung als `hist_pv_intermediate.parquet`.

In [ ]:
if _BUILD:
    df['weather_regime'] = (df['weather_code'].map(WEATHER_REGIME_MAP)
                              .astype(pd.CategoricalDtype(categories=ORDERED_CATEGORIES)))
    df['temperature_penalty_rel'] = ((df['temperature_2m'] + 273.15 - 173.15) / 2).astype('float32')
    df.drop(columns=['weather_code'], inplace=True, errors='ignore')
    print(f'  weather_regime:          {df["weather_regime"].value_counts().index[0]} am häufigsten '
          f'({df["weather_regime"].isna().sum()} NaN)')
    print(f'  temperature_penalty_rel: min={df["temperature_penalty_rel"].min():.1f} '
          f'max={df["temperature_penalty_rel"].max():.1f}')

    df.to_parquet(INTERMEDIATE, index=False)
    size_mb = INTERMEDIATE.stat().st_size / 1e6
    print(f'Intermediate gespeichert: {INTERMEDIATE} ({size_mb:.0f} MB)')
    print(f'Zeilen: {len(df):,} | Spots: {df["spot_uuid"].nunique()} | Spalten: {len(df.columns)}')
    del df; gc.collect()

print('Schritte 3-8: abgeschlossen.')

## 9. Panel-Tilt/Azimuth, kwp\_est, kwh\_norm

Panel-Neigung und -Ausrichtung werden aus dem `recent_parquet` übernommen (bekannte Spots). Für Spots ohne Werte wird `pv_inference` genutzt, um Tilt und Azimuth aus dem Einspeiseprofil zu schätzen.

`kwp_est` (installierte Peakleistung in kWp) wird aus dem 95%-Quantil von `kwh / ghi` bei ausreichender Einstrahlung (GHI > 100 W/m²) abgeleitet und auf [3, 30] kWp begrenzt.

`kwh_norm = kwh / kwp_est` normiert die Einspeisung auf die Anlagengröße — das Zielmerkmal im Training.

In [ ]:
import importlib, types

df = pd.read_parquet(INTERMEDIATE)
df['ts'] = pd.to_datetime(df['ts'])
df['spot_uuid'] = df['spot_uuid'].astype('category')
float_cols2 = df.select_dtypes('float64').columns.tolist()
if float_cols2: df[float_cols2] = df[float_cols2].astype('float32')
gc.collect()
print(f'Intermediate geladen: {len(df):,} Zeilen | {df["spot_uuid"].nunique()} Spots')

# Panel-Tilt/Azimuth aus recent Parquet
panel_cols = ['spot_uuid','panel_tilt','panel_unimodal_azimuth','panel_is_ew_split',
              'panel_bimodal_east_azimuth','panel_bimodal_west_azimuth',
              'panel_bimodal_east_fraction','panel_tilt_azi_confidence','kwp_est']
if RECENT_PARQUET.exists():
    try:
        recent_panel = pd.read_parquet(RECENT_PARQUET, columns=panel_cols).drop_duplicates('spot_uuid')
        df = df.merge(recent_panel, on='spot_uuid', how='left')
        n_known = df["spot_uuid"][df["panel_tilt"].notna()].nunique()
        print(f'panel_tilt aus recent Parquet: {n_known} Spots bekannt')
    except Exception as e:
        print(f'recent Parquet nicht geladen: {e}')
        for col in panel_cols[1:]: df[col] = np.nan
else:
    print('recent Parquet nicht gefunden -- alle Tilt-Werte werden inferiert.')
    for col in panel_cols[1:]: df[col] = np.nan

# Tilt-Inference fuer Spots ohne Werte
missing_tilt = df.loc[df['panel_tilt'].isna(), 'spot_uuid'].unique()
if len(missing_tilt) > 0:
    print(f'Tilt-Inference fuer {len(missing_tilt)} Spots...')
    _exp_path = _HERE / 'experiments/total_pv_output' if (_HERE / 'experiments').exists() else _BASE.parent / 'experiments/total_pv_output'
    sys.path.insert(0, str(_exp_path))
    try:
        from pv_inference import estimate_tilt_azimuth_all_plants
        df_miss = df[df['spot_uuid'].isin(missing_tilt) & (df['solar_elevation'] > 5)][
            ['spot_uuid','ts','kwh','ghi','dni','dhi','tsi','latitude','longitude']].copy()
        df_miss['ts'] = pd.to_datetime(df_miss['ts'], utc=True)
        df_miss['spot_uuid'] = df_miss['spot_uuid'].astype(str)
        inferred = estimate_tilt_azimuth_all_plants(
            df_miss, plant_id_col='spot_uuid', power_col='kwh',
            ghi_col='ghi', dni_col='dni', dhi_col='dhi', tsi_col='tsi',
            lat_col='latitude', lon_col='longitude', ts_col='ts', n_workers=N_WORKERS_TILT)
        inferred = inferred.rename(columns={
            'plant_id':'spot_uuid','tilt':'panel_tilt','azimuth':'panel_unimodal_azimuth',
            'azimuth_east':'panel_bimodal_east_azimuth','azimuth_west':'panel_bimodal_west_azimuth',
            'east_fraction':'panel_bimodal_east_fraction','is_ew_split':'panel_is_ew_split',
            'overlap_count':'panel_tilt_azi_confidence'})
        inf_cols = [c for c in panel_cols if c in inferred.columns and c != 'kwp_est']
        df = df.merge(inferred[inf_cols], on='spot_uuid', how='left', suffixes=('','_inf'))
        for col in [c for c in inf_cols if c != 'spot_uuid']:
            if col+'_inf' in df.columns:
                df[col] = df[col].fillna(df[col+'_inf'])
                df.drop(columns=[col+'_inf'], inplace=True)
        print(f'  Inference abgeschlossen.')
    except ImportError as e:
        print(f'  pv_inference nicht gefunden: {e} -- Tilt-Werte bleiben NaN')

for col in ['panel_bimodal_east_azimuth','panel_bimodal_west_azimuth','panel_bimodal_east_fraction']:
    df[col] = df[col].fillna(0.0).astype('float32')
df['panel_is_ew_split'] = df['panel_is_ew_split'].fillna(False)

# kwp_est + kwh_norm
print('kwp_est + kwh_norm...')
reliable = df[df['ghi'] > 100].copy()
reliable['kwh_per_ghi'] = reliable['kwh'] / reliable['ghi']
peak_eff = reliable.groupby('spot_uuid')['kwh_per_ghi'].quantile(0.95).rename('max_kwh_est')
kwp_est_new = (peak_eff * 850).clip(3, 30).round(2)
df = df.merge(kwp_est_new.reset_index(), on='spot_uuid', how='left')
if 'kwp_est' not in df.columns:
    df['kwp_est'] = df['max_kwh_est']
else:
    df['kwp_est'] = df['kwp_est'].fillna(df['max_kwh_est'])
df['kwh_norm'] = (df['kwh'] / (df['kwp_est'] + EPS)).astype('float32')
print(f'kwp_est: min={df["kwp_est"].min():.1f} max={df["kwp_est"].max():.1f} kWp')


## 10. Analog-Features (k=10, saisonale Naehe +-60 Tage)

**Kernidee:** Fuer jeden Messzeitpunkt werden k=10 historisch aehnliche Zeitpunkte gesucht.
Saisonale Naehe von +-60 Tagen wird als Gewicht eingesetzt.
Die Berechnung kann bei vielen Spots einige Stunden dauern (COMPUTE_ANALOGS=True).

**Distanz-Funktion:** Gewichtete euklidische Distanz auf Einstrahlungs- und Wetter-Features:

| Feature | Gewicht |
|---------|---------|
| `dni_frac` | 3.0 |
| `dhi_frac` | 3.0 |
| `ghi_magnitude` | 3.0 |
| `temperature_2m` | 0.5 |
| `wind_speed_10m` | 0.5 |

Einstrahlungs-Features erhalten 6x staerkeres Gewicht als Wetter, da sie die Solarproduktion direkt bestimmen.

**Berechnete Features:**

| Feature | Bedeutung |
|---------|-----------|
| `analog_kwh_1` | kWh des naechsten Analogs (absolut) |
| `analog_knn_mean` | Mittlere kWh der 10 naechsten Analoga |
| `scaled_analog_kwh_1` | Analog-kWh skaliert auf aktuelle Einstrahlung |
| `scaled_analog_knn_mean` | KNN-Mittel skaliert auf aktuelle Einstrahlung -- **staerkstes Feature** |
| `analog_kwh_1_norm` | Normiert durch kwp\_est |
| `scaled_analog_knn_mean_norm` | Normiert durch kwp\_est -- als NaN/Outlier-Fill im Training verwendet |
| `analog_dist_1` | Distanz zum naechsten Analog (Feature-Raum) |
| `analog_dist_gap` | Luecke zwischen naechstem und zweitnaechstem Analog |
| `analog_knn_std` | Streuung der k=10 Analog-Vorhersagen |
| `analog_age_hours` | Alter des naechsten Analogs in Stunden |
| `analog_available` | Flag: mindestens 1 Analog gefunden |
| `irr_mismatch` | Verhaeltnis aktueller zu Analog-Einstrahlung |


In [ ]:
if not COMPUTE_ANALOGS:
    print('COMPUTE_ANALOGS=False -- Analog-Features werden uebersprungen.')
    for col in ['analog_kwh_1','analog_kwh_1_norm','analog_knn_mean','analog_knn_mean_norm',
                'scaled_analog_kwh_1','scaled_analog_kwh_1_norm','scaled_analog_knn_mean',
                'analog_available','analog_dist_1','analog_dist_gap',
                'analog_knn_std','analog_age_hours','irr_mismatch']:
        df[col] = np.nan
else:
    _utils_path = _HERE / 'utils' if (_HERE / 'utils').exists() else _BASE.parent / 'utils'
    if _utils_path.exists():
        import types as _types
        def _load_mod(name, path):
            spec = importlib.util.spec_from_file_location(name, path)
            mod  = _types.ModuleType(name); mod.__spec__ = spec; mod.__package__ = 'utils'
            spec.loader.exec_module(mod); return mod
        _dtypes_mod = _load_mod('utils.dtypes',   _utils_path / 'dtypes.py')
        sys.modules['utils.dtypes'] = _dtypes_mod
        _feat_mod   = _load_mod('utils.features', _utils_path / 'features.py')
        compute_analog_features = _feat_mod.compute_analog_features
    else:
        raise ImportError(f'utils/ nicht gefunden unter {_utils_path}')

    print('Berechne Analog-Features...')
    df_analog = df[df['kwp_est'].notna()].copy()
    df_analog = compute_analog_features(
        df=df_analog, id_col='spot_uuid', ts_col='ts', target_col='kwh',
        distance_features=['dni_frac','dhi_frac','ghi_magnitude','temperature_2m','wind_speed_10m'],
        distance_weights=np.array([3.0, 3.0, 3.0, 0.5, 0.5]),
        season_proximity_days=SEASON_PROXIMITY_DAYS,
        elevation_threshold=SOLAR_ELEVATION_THRESHOLD,
        min_separation_hours=12.0, k=10, ghi_col='ghi',
        irr_ratio_clip=(0.3, 3.0), eps=EPS)
    analog_cols = ['spot_uuid','ts','analog_kwh_1','analog_knn_mean','analog_knn_std',
                   'analog_dist_1','analog_dist_gap','analog_age_hours','analog_available',
                   'scaled_analog_kwh_1','scaled_analog_knn_mean','irr_mismatch']
    df = df.merge(df_analog[analog_cols], on=['spot_uuid','ts'], how='left')
    df['analog_kwh_1_norm']        = (df['analog_kwh_1']        / (df['kwp_est'] + EPS)).astype('float32')
    df['scaled_analog_kwh_1_norm'] = (df['scaled_analog_kwh_1'] / (df['kwp_est'] + EPS)).astype('float32')
    df['analog_knn_mean_norm']     = (df['analog_knn_mean']      / (df['kwp_est'] + EPS)).astype('float32')
    df['scaled_analog_knn_mean_norm'] = (df['scaled_analog_knn_mean'] / (df['kwp_est'] + EPS)).astype('float32')
    print(f'Analog-Features berechnet. analog_available: {df["analog_available"].mean()*100:.1f}%')


## 11. Outlier-Entfernung

Drei Kriterien (identisch zu SPT_PV_D):
1. **Nacht-Erzeugung >5%**: solar_elevation < 3 deg aber kwh > 1
2. **Zu wenig Historie**: < 60 Tage Daten
3. **Zu geringe Gesamterzeugung**: < 30 kWh gesamt

Danach: Tages-Zeros mit hohem kt (kt > 0.25) -> NaN (echte Messluecke, nicht Nacht).


In [ ]:
n_before = df['spot_uuid'].nunique()

# 1. Spots mit >5% Nacht-Erzeugung
df['_night_bad'] = ((df['solar_elevation'] < 3) & (df['kwh'] > 1)).astype('int8')
spot_sizes    = df.groupby('spot_uuid', observed=True).size()
night_bad_sum = df.groupby('spot_uuid', observed=True)['_night_bad'].sum()
bad_night = (night_bad_sum / spot_sizes)[lambda s: s > 0.05].index
df.drop(columns=['_night_bad'], inplace=True)
df = df[~df['spot_uuid'].isin(bad_night)]
print(f'Nacht-Erzeugung >5%:      {len(bad_night):4d} Spots entfernt')

# 2. Zu wenig Historie (< 60 Tage = 5760 Zeilen bei 15-min)
n_rows = df.groupby('spot_uuid', observed=True).size()
bad_short = n_rows[n_rows < 5760].index
df = df[~df['spot_uuid'].isin(bad_short)]
print(f'Zu wenig Historie:        {len(bad_short):4d} Spots entfernt')

# 3. Zu geringe Gesamterzeugung (< 30 kWh)
total_kwh = df.groupby('spot_uuid', observed=True)['kwh'].sum()
bad_total = total_kwh[total_kwh < 30].index
df = df[~df['spot_uuid'].isin(bad_total)]
print(f'Zu geringe Erzeugung:     {len(bad_total):4d} Spots entfernt')

# Tages-Zeros bei hohem kt -> NaN
day_mask = (df['kwh'] == 0) & (df['kt'] > 0.25) & (df['solar_elevation'] > 3)
df.loc[day_mask, 'kwh']      = np.nan
df.loc[day_mask, 'kwh_norm'] = np.nan
print(f'Tages-Zeros zu NaN:       {day_mask.sum():,} Zeilen')

n_after = df['spot_uuid'].nunique()
print(f'\nSpots: {n_before} -> {n_after} ({n_before - n_after} entfernt)')
print(f'Zeilen: {len(df):,}')
df['spot_uuid'] = df['spot_uuid'].cat.remove_unused_categories()


## 12. Feature-Auswahl & Speichern als `data_final.parquet`

In [ ]:
# Zeit-Features falls noch nicht vorhanden (Sicherheit)
if 'month_sin' not in df.columns:
    h = df['ts'].dt.hour + df['ts'].dt.minute / 60
    df['hour_sin']  = np.sin(2*np.pi*h/24).astype('float32')
    df['hour_cos']  = np.cos(2*np.pi*h/24).astype('float32')
    df['month_sin'] = np.sin(2*np.pi*df['ts'].dt.month/12).astype('float32')
    df['month_cos'] = np.cos(2*np.pi*df['ts'].dt.month/12).astype('float32')
    df['doy_sin']   = np.sin(2*np.pi*df['ts'].dt.dayofyear/365).astype('float32')
    df['doy_cos']   = np.cos(2*np.pi*df['ts'].dt.dayofyear/365).astype('float32')

available = [c for c in FINAL_FEATURES if c in df.columns]
missing   = [c for c in FINAL_FEATURES if c not in df.columns]
if missing:
    print(f'Fehlende Spalten (werden nicht gespeichert): {missing}')

df_out = df[available].copy()
df_out['spot_uuid'] = df_out['spot_uuid'].astype('category')

print(f'Speichere {len(df_out):,} Zeilen | {df_out["spot_uuid"].nunique()} Spots | {len(available)} Features')
df_out.to_parquet(OUT_PARQUET, index=False, compression='snappy')
size_gb = OUT_PARQUET.stat().st_size / 1e9
print(f'Gespeichert: {OUT_PARQUET} ({size_gb:.2f} GB)')
print(f'\nFeatures:')
for col in available:
    print(f'  {col}')


## 13. Validierung

In [ ]:
import pyarrow.dataset as _pads
import pyarrow.parquet as _pq

print(f'data_final.parquet: {OUT_PARQUET.stat().st_size/1e9:.2f} GB')
_df = _pads.dataset(str(OUT_PARQUET)).to_table(
    columns=['ts','spot_uuid','kwh_norm','kwp_est']).to_pandas()
_df['ts'] = pd.to_datetime(_df['ts'])
print(f'Zeilen:   {len(_df):,}')
print(f'Spots:    {_df["spot_uuid"].nunique():,}')
print(f'Zeitraum: {_df["ts"].min()} bis {_df["ts"].max()}')
print(f'kwh_norm: min={_df["kwh_norm"].min():.4f} | max={_df["kwh_norm"].max():.4f} | NaN={_df["kwh_norm"].isna().sum():,}')
print(f'kwp_est:  min={_df["kwp_est"].min():.1f} | max={_df["kwp_est"].max():.1f} kWp')
del _df
print('Validierung abgeschlossen.')
